# 03 · Evaluate: base Gemma vs fine-tuned Gemma vs Gemini

All three models answer the same **48 held-out test conversations** (unseen respondents and unseen message templates).

| Metric | What it measures |
|---|---|
| JSON valid | extraction is parseable JSON |
| Intent accuracy | fee check / rent question / off-topic recognised correctly |
| Field accuracy | area, house type, language and the five fee amounts correct |
| Fees exact | every fee amount (incl. other fees) exactly right |
| Verdict accuracy | after the fee engine runs, each fee gets the right amount and verdict |
| Grounded replies | every number in the reply appears in the engine result |
| Fee coverage | every fee is mentioned with its amount |
| Language match | English vs Pidgin reply as expected |

**Runs on:** Colab T4 GPU, about 10–20 minutes. Gemini is optional (needs `GEMINI_API_KEY` in Colab Secrets).

In [ ]:
# --- Setup: get the project code (works in Colab and locally) ---
REPO_URL = "https://github.com/YOUR-GITHUB-USERNAME/fairrent-akure.git"  # <-- change to your repo

import os, sys
if os.path.exists("../src/fairrent"):          # running locally from notebooks/
    os.chdir("..")
elif not os.path.exists("src/fairrent"):       # running in Colab
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(folder):
        os.system(f"git clone -q {REPO_URL}")
    os.chdir(folder)
sys.path.insert(0, "src")
print("Working in:", os.getcwd())

In [ ]:
!pip install -q "transformers>=4.50" "peft>=0.13" accelerate google-genai tabulate

In [ ]:
import json, time
import pandas as pd
import torch
from huggingface_hub import login

try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("Using existing Hugging Face login.", e)

from fairrent.engine import FeeEngine
from fairrent.evaluate import (normalise_extraction, parse_json, score_extraction,
                               score_reply, score_verdicts, summarise)
from fairrent.llm import DEFAULT_MODEL_ID, generate, load_model
from fairrent.prompts import EXTRACT_SYSTEM, REPLY_SYSTEM, extract_messages, reply_messages

tenants = pd.read_csv("data/tenants_clean.csv", keep_default_na=False, na_values=[""])
engine = FeeEngine(tenants)
test = [json.loads(l) for l in open("data/conversations_test.jsonl", encoding="utf-8")]
print(len(test), "test conversations")

## 1. Load Gemma with the LoRA adapter

Point `ADAPTER_PATH` at the adapter from notebook 02 (in this session, on Google Drive, or on Hugging Face).

In [ ]:
ADAPTER_PATH = "fairrent-gemma3-1b-lora"
if not os.path.exists(ADAPTER_PATH):
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER_PATH = "/content/drive/MyDrive/fairrent/fairrent-gemma3-1b-lora"

model, tokenizer = load_model(DEFAULT_MODEL_ID, ADAPTER_PATH)
print("Loaded base model + adapter from", ADAPTER_PATH)

## 2. Run the models

The **reply** task gets the gold engine result as input, so reply quality is measured on its own.
The **verdict** score runs each model's own extraction through the engine (end to end).

In [ ]:
extract_prompts = [extract_messages(c["message"]) for c in test]
reply_prompts = [reply_messages(c["result"]) for c in test]

def run_gemma(use_adapter: bool):
    start = time.time()
    if use_adapter:
        ext = generate(model, tokenizer, extract_prompts, max_new_tokens=200)
        rep = generate(model, tokenizer, reply_prompts, max_new_tokens=320)
    else:
        with model.disable_adapter():
            ext = generate(model, tokenizer, extract_prompts, max_new_tokens=200)
            rep = generate(model, tokenizer, reply_prompts, max_new_tokens=320)
    print(f"{'fine-tuned' if use_adapter else 'base'} Gemma: {(time.time()-start)/60:.1f} min")
    return ext, rep

predictions = {}
predictions["Gemma 3 1B (base)"] = run_gemma(False)
predictions["Gemma 3 1B + LoRA (ours)"] = run_gemma(True)

### Optional: Gemini baseline

Add `GEMINI_API_KEY` in Colab Secrets (free key from https://aistudio.google.com). Skip this cell if you don't have one.
Gemini gets the same instructions but no fine-tuning. It makes 96 calls with a short pause between them (about 15 minutes) to stay within free-tier limits.

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash"   # change if a newer Flash model is available
try:
    from google import genai
    from google.genai import types
    from google.colab import userdata
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

    available = [m.name.split("/")[-1] for m in client.models.list()]
    assert GEMINI_MODEL in available, f"{GEMINI_MODEL} not available. Flash models: {[m for m in available if 'flash' in m]}"

    def ask_gemini(messages):
        system, user = messages[0]["content"], messages[1]["content"]
        last = None
        for attempt in range(5):
            try:
                r = client.models.generate_content(
                    model=GEMINI_MODEL, contents=user,
                    config=types.GenerateContentConfig(system_instruction=system, temperature=0))
                time.sleep(7)                      # stay under the free-tier rate limit
                return r.text or ""
            except Exception as e:
                last = e
                time.sleep(15 * (attempt + 1))
        print("Gemini call failed:", last)
        return None

    start = time.time()
    g_ext = [ask_gemini(m) for m in extract_prompts]
    g_rep = [ask_gemini(m) for m in reply_prompts]
    n_fail = sum(x is None for x in g_ext + g_rep)
    if n_fail:
        raise RuntimeError(f"{n_fail}/{len(g_ext + g_rep)} Gemini calls failed (quota?). "
                           "Try GEMINI_MODEL = 'gemini-2.5-flash-lite' or run later.")
    predictions[f"Gemini ({GEMINI_MODEL})"] = (g_ext, g_rep)
    print(f"Gemini: {(time.time()-start)/60:.1f} min")
except Exception as e:
    print("Skipping Gemini baseline:", e)

## 3. Score

In [ ]:
rows, table = [], []
for name, (ext_out, rep_out) in predictions.items():
    per = []
    for c, e_raw, r_raw in zip(test, ext_out, rep_out):
        pred = parse_json(e_raw)
        s = score_extraction(pred, c["extraction"])
        norm = normalise_extraction(pred)
        s["verdict_acc"] = score_verdicts(engine.check(norm) if norm else None, c["result"])
        r = score_reply(r_raw, c["result"])
        s.update({k: r[k] for k in ["grounded", "coverage", "language_ok"]})
        per.append(s)
        rows.append({"model": name, "id": c["id"], "source": c["source"],
                     "language": c["extraction"]["language"], "message": c["message"],
                     "extraction_output": e_raw, "reply_output": r_raw,
                     "ungrounded_numbers": r["ungrounded_numbers"], **s})
    table.append({"Model": name, **summarise(per)})

results = pd.DataFrame(table).set_index("Model")
nice = results.rename(columns={
    "json_valid": "JSON valid", "intent_acc": "Intent acc.", "field_acc": "Field acc.",
    "fees_exact": "Fees exact", "exact_match": "Exact match", "verdict_acc": "Verdict acc.",
    "grounded": "Grounded replies", "coverage": "Fee coverage", "language_ok": "Language match"})
(nice * 100).round(1)

In [ ]:
os.makedirs("results", exist_ok=True)
pd.DataFrame(rows).to_csv("results/test_predictions.csv", index=False)
json.dump(table, open("results/eval_results.json", "w"), indent=2)
md = (nice * 100).round(1).to_markdown()
open("results/eval_table.md", "w").write(md + "\n\nAll values are percentages on the 48 held-out test conversations.\n")
print(md)
print("\nPaste this table into MODEL_CARD.md under 'Evaluation results'.")

## 4. Results by language and by kind of message

In [ ]:
df = pd.DataFrame(rows)
by_lang = df.groupby(["model", "language"])[["field_acc", "verdict_acc", "grounded"]].mean().mul(100).round(1)
by_lang

In [ ]:
df["kind"] = df["source"].str.split(":").str[0]
df.groupby(["model", "kind"])[["exact_match", "verdict_acc"]].mean().mul(100).round(1)

## 5. Error analysis

Look at the fine-tuned model's mistakes. Use what you find in the model card's *Limitations* section.

In [ ]:
ours = df[df.model.str.contains("LoRA")]
for _, r in ours[ours.exact_match == 0].head(8).iterrows():
    print(f"[{r.id} | {r.language}] {r.message}")
    print("  model:", r.extraction_output[:300])
    print()
bad = ours[ours.grounded == 0]
print(f"Ungrounded replies: {len(bad)}")
for _, r in bad.head(5).iterrows():
    print(r.id, r.ungrounded_numbers)